## Namibia 2000 DHS Salt analysis
### 22 October 2020

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)

from aw_analytics import mean_wt

In [3]:
fn = 'NMHR41FL.DTA'
path = Path.cwd().joinpath('data', fn)

df = pd.read_stata(path)
df.shape

(6392, 1646)

In [4]:
# change dtypes to string
str_cols = ['hv015', 'hv023', 'hv024', 'hv025']
df[str_cols] = df[str_cols].astype('string')

In [5]:
# Create regular weight
df['wt'] = df['hv005'] / 1000000

In [6]:
# Type of salt by region

var = 'sh35'
ind_vars = ['hv015', 'hv025', 'hv024']

var_df = pd.get_dummies(df[var])
var_col_names = var_df.columns.to_list()
ind_df = df[ind_vars]
wt_df = df['wt']


temp = ind_df[:].join(var_df).join(wt_df)

# List comp to get mean values
mean_list = [temp.groupby(ind_vars[i]).apply(mean_wt, var_col_names[j], 'wt') for i in range(len(ind_vars)) for j in range(len(var_col_names))]
count_list = [temp.groupby(ind_vars[i])['wt'].apply(sum).round(1) for i in range(len(ind_vars))]

# Concat lists
var_len = len(var_col_names)
mean_concat = [pd.concat(mean_list[(i*var_len):(i*var_len+var_len)], axis=1) for i in range(len(ind_vars))]
count_concat = [pd.concat([count_list[i]], axis=0) for i in range(len(ind_vars))]

# Concat (vertical stack)
mean_df = pd.concat(mean_concat, axis=0)
count_df = pd.DataFrame(pd.concat(count_concat, axis=0))

# Rename cols
old_names = [i for i in range(len(var_col_names))]
col_dict = dict(zip(old_names, var_col_names))
mean_df = mean_df.rename(columns = col_dict)

# Join count to df
output_df = mean_df.join(count_df)
output_df = output_df.rename(columns={'wt': 'Weighted_Count'})

output_df.to_csv('./output/2000_salt_type.csv')

In [7]:
# Source of salt by region
var = 'sh36'
ind_vars = ['hv015', 'hv025', 'hv024']


var_df = pd.get_dummies(df[var])
var_col_names = var_df.columns.to_list()
ind_df = df[ind_vars]
wt_df = df['wt']


temp = ind_df[:].join(var_df).join(wt_df)

# List comp to get mean values
mean_list = [temp.groupby(ind_vars[i]).apply(mean_wt, var_col_names[j], 'wt') for i in range(len(ind_vars)) for j in range(len(var_col_names))]
count_list = [temp.groupby(ind_vars[i])['wt'].apply(sum).round(1) for i in range(len(ind_vars))]

# Concat lists
var_len = len(var_col_names)
mean_concat = [pd.concat(mean_list[(i*var_len):(i*var_len+var_len)], axis=1) for i in range(len(ind_vars))]
count_concat = [pd.concat([count_list[i]], axis=0) for i in range(len(ind_vars))]

# Concat (vertical stack)
mean_df = pd.concat(mean_concat, axis=0)
count_df = pd.DataFrame(pd.concat(count_concat, axis=0))

# Rename cols
old_names = [i for i in range(len(var_col_names))]
col_dict = dict(zip(old_names, var_col_names))
mean_df = mean_df.rename(columns = col_dict)

# Join count to df
output_df = mean_df.join(count_df)
output_df = output_df.rename(columns={'wt': 'Weighted_Count'})

output_df.to_csv('./output/2000_salt_source.csv')

In [8]:
df.hv234.value_counts(dropna=False)

15+ ppm              3331
0 ppm (no iodine)    1148
< 15 ppm             1147
no salt               634
NaN                   132
Name: hv234, dtype: int64

In [9]:
df['salt_tested'] = np.where(df['hv234'] == 'no salt', np.nan, df['hv234'])
df.salt_tested.value_counts(dropna=False)
df = df.dropna(subset=['salt_tested'])
df.shape

(5626, 1648)

In [10]:
df.salt_tested.value_counts(dropna=False)

15+ ppm              3331
0 ppm (no iodine)    1148
< 15 ppm             1147
Name: salt_tested, dtype: int64

In [11]:
# Iodine coverage by region
var = 'salt_tested'
ind_vars = ['hv015', 'hv025', 'hv024']


var_df = pd.get_dummies(df[var], prefix=str(var)).replace({np.nan: 0})
var_col_names = var_df.columns.to_list()
ind_df = df[ind_vars]
wt_df = df['wt']


temp = ind_df[:].join(var_df).join(wt_df)

# List comp to get mean values
mean_list = [temp.groupby(ind_vars[i]).apply(mean_wt, var_col_names[j], 'wt') for i in range(len(ind_vars)) for j in range(len(var_col_names))]
count_list = [temp.groupby(ind_vars[i])['wt'].apply(sum).round(1) for i in range(len(ind_vars))]

# Concat lists
var_len = len(var_col_names)
mean_concat = [pd.concat(mean_list[(i*var_len):(i*var_len+var_len)], axis=1) for i in range(len(ind_vars))]
count_concat = [pd.concat([count_list[i]], axis=0) for i in range(len(ind_vars))]

# Concat (vertical stack)
mean_df = pd.concat(mean_concat, axis=0)
count_df = pd.DataFrame(pd.concat(count_concat, axis=0))

# Rename cols
old_names = [i for i in range(len(var_col_names))]
col_dict = dict(zip(old_names, var_col_names))
mean_df = mean_df.rename(columns = col_dict)

# Join count to df
output_df = mean_df.join(count_df)
output_df = output_df.rename(columns={'wt': 'Weighted_Count'})

output_df.to_csv('./output/2000_iodine.csv')